In [2]:
from typing import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.types import RetryPolicy
from loguru import  logger
from requests import HTTPError


#1. 状態を宣言
class EmptyState(TypedDict):
    pass
#2. ノードを宣言
def node_a(state:EmptyState) -> EmptyState:
    logger.info("node a が実行中")
    raise HTTPError

#3. グラフを構築
builder = StateGraph(state_schema = EmptyState)
builder.add_node(
    "node_a",
    node_a,
    retry_policy=RetryPolicy(
        max_attempts=3,
        jitter = False,
        max_interval=128
    ))

builder.add_edge(START,"node_a")
builder.add_edge("node_a",END)
graph = builder.compile()

try:
    graph.invoke({})
except HTTPError as e:
    logger.info("リトライ回数を使い切りました:{}",e)



2026-08-14 17:03:33.568 | INFO     | __main__:node_a:14 - node a が実行中
2026-08-14 17:03:34.069 | INFO     | __main__:node_a:14 - node a が実行中
2026-08-14 17:03:35.070 | INFO     | __main__:node_a:14 - node a が実行中
2026-08-14 17:03:35.071 | INFO     | __main__:<module>:35 - リトライ回数を使い切りました:
